# Debugging step by step

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Debugging mit PyCharm</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kursinhalt &nbsp;|&nbsp; Werkzeuge</p>
</div>
</div>

**Legende**

> **[Kursinhalt]** Dieses Notebook ist kein PCAP-Pruefungsinhalt

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Was ist Debugging -- und was ist es nicht?
</span>
</div>

Ein Programm produziert ein falsches Ergebnis. Oder es stuerzt ab. Oder es haengt. Man weiss dass etwas nicht stimmt -- aber nicht warum.

An diesem Punkt helfen zwei verschiedene Werkzeuge:

**Testen** beantwortet die Frage: *Was ist kaputt?* Ein Test vergleicht das tatsaechliche Ergebnis mit dem erwarteten. Wenn sie nicht uebereinstimmen, ist der Test rot. Man weiss jetzt dass eine bestimmte Funktion falsch rechnet -- aber noch nicht warum.

**Debugging** beantwortet die Frage: *Warum ist es kaputt?* Der Debugger haelt das laufende Programm an einem bestimmten Punkt an und zeigt den vollstaendigen inneren Zustand: welche Variablen welche Werte haben, wo im Code man sich befindet, wie man dorthin gelangt ist.

Testen und Debugging ergaenzen sich:

```
Test schlaegt fehl
      |
      v
Debugger: Wo genau passiert der Fehler? Was haben die Variablen in diesem Moment?
      |
      v
Bug gefunden und behoben
      |
      v
Test laeuft durch -- Bestaetigung dass der Fix korrekt ist
```

Der Unterschied zum einfachen `print()`-Debugging: Ein `print()` muss man nach dem Debuggen wieder entfernen, veraendert den Code, und zeigt nur was man vorher schon fuer relevant gehalten hat. Der Debugger zeigt alles, veraendert nichts, und kann jederzeit an beliebigen Stellen angehalten werden.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Das Beispielprogramm -- mit eingebautem Bug
</span>
</div>

Wir arbeiten mit einem Warenkorb-System. Es berechnet den Gesamtpreis einer Bestellung inklusive Mengenrabatt. Das Programm laeuft ohne Fehler -- aber die Endrechnung stimmt nicht.

Speichere diesen Code als `warenkorb.py` in PyCharm:

In [ ]:
# warenkorb.py

def berechne_zwischensumme(artikel):
    """Summiert die Preise aller Artikel."""
    gesamt = 0
    for name, preis, menge in artikel:
        gesamt += preis * menge
    return gesamt


def berechne_rabatt(zwischensumme, rabatt_prozent):
    """Berechnet den Rabattbetrag."""
    # BUG: Hier steckt der Fehler
    return zwischensumme / 100 * rabatt_prozent


def berechne_steuer(betrag, steuersatz=19):
    """Berechnet die Mehrwertsteuer."""
    return betrag / 100 * steuersatz


def berechne_gesamtpreis(artikel, rabatt_prozent=0):
    """Berechnet den Gesamtpreis inkl. Rabatt und MwSt."""
    zwischensumme = berechne_zwischensumme(artikel)
    rabatt        = berechne_rabatt(zwischensumme, rabatt_prozent)
    netto         = zwischensumme - rabatt
    steuer        = berechne_steuer(netto)
    gesamt        = netto + steuer
    return round(gesamt, 2)


if __name__ == '__main__':
    bestellung = [
        ('Python Buch',    29.99,  2),
        ('USB-Hub',        19.90,  1),
        ('Tastatur',       89.00,  1),
    ]

    # 10% Rabatt weil Stammkunde
    preis = berechne_gesamtpreis(bestellung, rabatt_prozent=10)
    print(f'Gesamtpreis: {preis} EUR')

    # Erwartetes Ergebnis:
    # Zwischensumme: 29.99*2 + 19.90 + 89.00 = 168.87
    # Rabatt 10%:    168.87 * 0.10 = 16.887
    # Netto:         168.87 - 16.887 = 151.983
    # MwSt 19%:      151.983 * 0.19 = 28.877
    # Gesamt:        151.983 + 28.877 = 180.86
    print(f'Erwartet:    180.86 EUR')

Das Programm laeuft ohne Fehlermeldung -- aber die Ausgabe stimmt nicht mit dem erwarteten Wert ueberein. Es gibt keinen `NameError`, keinen `TypeError`, keinen Absturz. Das ist die schwierigste Art von Bug: stiller Rechenfehler.

Jetzt setzen wir den Debugger ein um herauszufinden wo die Rechnung abweicht.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Schritt 1 -- Breakpoint setzen
</span>
</div>

Ein **Breakpoint** ist eine Markierung in der IDE -- kein Code, kein `print()`. Er sagt dem Debugger: *Halte das Programm genau hier an, bevor diese Zeile ausgefuehrt wird.*

**In PyCharm einen Breakpoint setzen:**
Einfach links neben die Zeilennummer klicken -- dort wo der graue Rand ist. Ein roter Punkt erscheint. Nochmals klicken entfernt ihn.

**Wo setzen wir den ersten Breakpoint?**

Wir wissen dass das Ergebnis falsch ist, aber nicht wo. Eine sinnvolle Strategie: Breakpoint in `berechne_gesamtpreis()` auf die Zeile `zwischensumme = berechne_zwischensumme(artikel)` -- damit sehen wir alle Zwischenwerte nacheinander.

Setze einen Breakpoint auf diese Zeile in `warenkorb.py`:
```
zwischensumme = berechne_zwischensumme(artikel)   # <-- Breakpoint hier
```

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Schritt 2 -- Debug-Modus starten
</span>
</div>

Das Programm nicht mit dem gruenen Play-Pfeil starten -- das waere der normale Run-Modus, der Breakpoints ignoriert.

Stattdessen:
- Das **Kaefer-Symbol** (Bug-Icon) neben dem Play-Pfeil
- Oder: Rechtsklick auf die Datei -> *Debug 'warenkorb'*
- Oder: **Shift+F9**

Das Programm startet, laeuft normal -- und haelt am Breakpoint an. Die betreffende Zeile wird blau hervorgehoben. Unten oeffnet sich das **Debug-Fenster**.

Das Debug-Fenster hat zwei wichtige Bereiche:

**Links -- Frames (Call Stack):** Zeigt wo man sich gerade befindet. Ganz oben ist die aktuelle Zeile, darunter die Funktion die diese Funktion aufgerufen hat, darunter die naechste, und so weiter. Man kann auf eine Zeile im Stack klicken um den Zustand an diesem Punkt zu sehen.

**Rechts -- Variables:** Zeigt alle Variablen die im aktuellen Geltungsbereich sichtbar sind -- mit ihren aktuellen Werten. Objekte koennen aufgeklappt werden um ihre Attribute zu sehen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Schritt 3 -- durch den Code navigieren
</span>
</div>

Das Programm ist pausiert. Jetzt kann man es Zeile fuer Zeile vorwaerts bewegen. Dafuer gibt es vier Befehle:

**Step Over -- F8**
Fuehrt die aktuelle Zeile aus und haelt bei der naechsten. Wenn die Zeile einen Funktionsaufruf enthaelt, wird die Funktion komplett ausgefuehrt -- man sieht nicht was darin passiert, nur das Ergebnis.

Wann benutzen: Wenn man einer Funktion vertraut und nur das Ergebnis sehen will.

**Step Into -- F7**
Springt in den Funktionsaufruf der aktuellen Zeile hinein. Man landet in der ersten Zeile der aufgerufenen Funktion und sieht was dort passiert.

Wann benutzen: Wenn man eine Funktion verdaechtig findet und sehen will was sie intern macht.

**Step Out -- Shift+F8**
Fuehrt den Rest der aktuellen Funktion aus und springt zurueck zur aufrufenden Stelle. Nuetzlich wenn man in einer Funktion steckt die man nicht weiter verfolgen will.

**Resume -- F9**
Laeuft bis zum naechsten Breakpoint weiter -- oder bis das Programm endet wenn kein weiterer Breakpoint gesetzt ist.

Wann benutzen: Wenn man genug gesehen hat und zum naechsten Haltepunkt springen will.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Den Bug finden -- Schritt fuer Schritt
</span>
</div>

Das Programm haelt am Breakpoint in `berechne_gesamtpreis()`. Der Variables-Bereich zeigt:
- `artikel` -- die Liste der drei Produkte
- `rabatt_prozent` -- der Wert 10

**F8 -- Step Over:** Die Zeile `zwischensumme = berechne_zwischensumme(artikel)` wird ausgefuehrt. Variables zeigt jetzt:
- `zwischensumme` = 168.87

Das ist korrekt: 29.99 * 2 + 19.90 + 89.00 = 168.87.

**F8 -- Step Over:** Naechste Zeile: `rabatt = berechne_rabatt(zwischensumme, rabatt_prozent)`. Wir koennen schon vermuten dass hier etwas schieflaeuft -- druecken wir diesmal **F7 -- Step Into** statt F8 um die Funktion zu betreten.

Wir sind jetzt in `berechne_rabatt()`. Variables zeigt:
- `zwischensumme` = 168.87
- `rabatt_prozent` = 10

**F8 -- Step Over:** Die Berechnung wird ausgefuehrt. Der Rueckgabewert ist 16.887. Das stimmt!

Wir kehren zurueck zu `berechne_gesamtpreis()`. `rabatt` = 16.887.

**F8 -- Step Over:** `netto = zwischensumme - rabatt` = 168.87 - 16.887 = 151.983. Korrekt.

**F8 -- Step Into** bei `steuer = berechne_steuer(netto)`. Variables:
- `betrag` = 151.983
- `steuersatz` = 19

Berechnung: `151.983 / 100 * 19` = 28.877. Korrekt.

Zurueck in `berechne_gesamtpreis()`: `gesamt = netto + steuer` = 151.983 + 28.877 = 180.86.

Moment -- das Ergebnis ist doch korrekt? Dann liegt der Bug woanders. Lass uns das Programm nochmals starten und diesmal die Zwischensumme genauer pruefen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
7. Watches und Evaluate Expression
</span>
</div>

**Watches** -- bestimmte Ausdruecke dauerhaft beobachten

Im Variables-Bereich gibt es einen Reiter *Watches*. Hier kann man beliebige Python-Ausdruecke eintragen die bei jedem Schritt neu berechnet und angezeigt werden. Zum Beispiel:
- `zwischensumme - rabatt` -- zeigt den Nettobetrag waehrend man noch in der Funktion ist
- `len(artikel)` -- die Anzahl Artikel
- `artikel[0][1] * artikel[0][2]` -- der Preis des ersten Artikels

**Evaluate Expression -- Alt+F8**

Ein Dialogfenster in dem man waehrend der Pause beliebigen Python-Code ausfuehren kann -- im Kontext des aktuellen Zustands. Man kann Variablen inspizieren, Berechnungen testen, Methoden aufrufen:

```python
# Im Evaluate-Dialog:
zwischensumme * 0.10          # Was waere der korrekte Rabatt?
sum(p * m for _, p, m in artikel)  # Zwischensumme anders berechnet
berechne_rabatt(168.87, 10)   # Funktion direkt aufrufen
```

Das ist ein sehr maechiges Werkzeug -- man kann Hypothesen direkt im laufenden Programm testen ohne den Code zu veraendern.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
8. Conditional Breakpoints -- nur anhalten wenn eine Bedingung gilt
</span>
</div>

Manchmal will man nicht bei jedem Durchlauf anhalten -- nur wenn eine bestimmte Bedingung erfuellt ist. Zum Beispiel: In einer Schleife ueber 1000 Artikel will man nur anhalten wenn der Preis eines Artikels ueber 100 liegt.

**Conditional Breakpoint setzen:**
1. Rechtsklick auf den roten Breakpoint-Punkt
2. Im Dialog erscheint ein Feld *Condition*
3. Dort einen Python-Ausdruck eingeben: z.B. `preis > 100`
4. Das Programm haelt nur an diesem Breakpoint wenn die Bedingung `True` ergibt

In unserem Warenkorb-Beispiel koennte man in `berechne_zwischensumme()` einen Conditional Breakpoint in der Schleife setzen:

```
for name, preis, menge in artikel:   # <-- Breakpoint mit Condition: preis > 50
    gesamt += preis * menge
```

Das Programm haelt dann nur bei der Tastatur (89.00 EUR) an -- nicht bei den anderen Artikeln.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
9. Der Bug -- Aufloesung
</span>
</div>

Der Bug in `warenkorb.py` ist subtil. Schau dir `berechne_rabatt()` nochmal an:

```python
def berechne_rabatt(zwischensumme, rabatt_prozent):
    return zwischensumme / 100 * rabatt_prozent
```

Das sieht korrekt aus -- und fuer unseren Fall ergibt `168.87 / 100 * 10 = 16.887`. Das stimmt.

Aber schau dir `berechne_steuer()` an:

```python
def berechne_steuer(betrag, steuersatz=19):
    return betrag / 100 * steuersatz
```

Auch korrekt -- `151.983 / 100 * 19 = 28.877`.

Das Ergebnis 180.86 ist also tatsaechlich korrekt. Das bedeutet: Das "erwartete" Ergebnis im Code ist falsch berechnet. Das ist ein haeufiges Szenario beim Debugging -- manchmal liegt der Fehler nicht im Produktionscode, sondern in der eigenen Erwartung.

Probiere jetzt eine andere Version mit einem echten Bug:

In [ ]:
# warenkorb_bug.py -- diesmal mit echtem Bug

def berechne_zwischensumme(artikel):
    gesamt = 0
    for name, preis, menge in artikel:
        gesamt += preis * menge
    return gesamt


def berechne_rabatt(zwischensumme, rabatt_prozent):
    # BUG: Rabatt wird auf den falschen Wert berechnet
    # rabatt_prozent ist z.B. 10 (fuer 10%)
    # aber hier wird durch rabatt_prozent dividiert statt durch 100
    return zwischensumme / rabatt_prozent   # FALSCH: gibt 16.887 statt 16.887
    # Hoppla -- fuer 10% stimmt das zufaellig!
    # Aber fuer 20% Rabatt: 168.87 / 20 = 8.44 statt 33.77


def berechne_steuer(betrag, steuersatz=19):
    return betrag / 100 * steuersatz


def berechne_gesamtpreis(artikel, rabatt_prozent=0):
    zwischensumme = berechne_zwischensumme(artikel)
    rabatt        = berechne_rabatt(zwischensumme, rabatt_prozent)
    netto         = zwischensumme - rabatt
    steuer        = berechne_steuer(netto)
    gesamt        = netto + steuer
    return round(gesamt, 2)


if __name__ == '__main__':
    bestellung = [
        ('Python Buch',    29.99,  2),
        ('USB-Hub',        19.90,  1),
        ('Tastatur',       89.00,  1),
    ]

    # Mit 10% Rabatt -- zufaellig korrekt wegen des Bugs!
    preis_10 = berechne_gesamtpreis(bestellung, rabatt_prozent=10)
    print(f'10% Rabatt: {preis_10} EUR  (erwartet: 180.86)')

    # Mit 20% Rabatt -- hier zeigt sich der Bug deutlich
    preis_20 = berechne_gesamtpreis(bestellung, rabatt_prozent=20)
    print(f'20% Rabatt: {preis_20} EUR  (erwartet: 161.34)')

Das ist ein klassisches Beispiel eines **latenten Bugs**: Fuer bestimmte Eingaben (10%) liefert der Code zufaellig das richtige Ergebnis -- fuer andere (20%) nicht. Solche Bugs sind besonders gefaehrlich weil sie lange unentdeckt bleiben.

**Debugging-Auftrag:** Setze in PyCharm einen Breakpoint in `berechne_rabatt()` und verfolge den Wert von `zwischensumme / rabatt_prozent` bei `rabatt_prozent=20`. Nutze Evaluate Expression um die korrekte Berechnung zu testen: `zwischensumme / 100 * rabatt_prozent`.

---

**Zusammenfassung**

| Konzept | Erklaerung |
|---------|------------|
| Debugging vs. Testen | Testen findet *was* kaputt ist -- Debugging findet *warum* |
| Breakpoint | Markierung in der IDE -- das Programm pausiert vor dieser Zeile |
| Debug-Modus starten | Kaefer-Symbol oder Shift+F9 -- nicht der normale Run |
| Variables-Bereich | Zeigt alle sichtbaren Variablen mit aktuellen Werten |
| Frames (Call Stack) | Zeigt wo man sich befindet und wie man dort hingekommen ist |
| Step Over (F8) | Naechste Zeile -- Funktionsaufrufe werden komplett ausgefuehrt |
| Step Into (F7) | In einen Funktionsaufruf hineinspringen |
| Step Out (Shift+F8) | Rest der aktuellen Funktion ausfuehren und zurueckspringen |
| Resume (F9) | Bis zum naechsten Breakpoint weiterlaufen |
| Watches | Ausdruecke dauerhaft beobachten waehrend man durch den Code geht |
| Evaluate (Alt+F8) | Beliebigen Python-Code im Kontext des Programms ausfuehren |
| Conditional Breakpoint | Nur anhalten wenn eine Bedingung wahr ist |
| Latenter Bug | Bug der nur bei bestimmten Eingaben auftritt -- besonders gefaehrlich |